# Legacy checkpoint evaluation

Evaluates the legacy 8-expert checkpoint — the model behind the 0.0168 / 0.9151 result — with
the **current** evaluation code. This is a diagnostic, not a result:

- reads **~0.0168** → the current pipeline reproduces the old number, so the gap to the new
  runs' ~0.0195 is a training-side difference;
- reads **~0.019** → the evaluation protocol changed, and the current numbers are the
  consistent ones.

**Before running:** upload the checkpoint to Google Drive, copy its file ID, and paste it into
`LEGACY_DRIVE_ID` in the next cell. Nothing else needs editing.


In [ ]:
import os
import json
import hashlib
import shutil
import subprocess
import sys

# ------------------------------------------------------------------------------
# Paste the Google Drive file ID of the legacy checkpoint here.
# For a link like  https://drive.google.com/file/d/1AbC...XyZ/view?usp=sharing
# the ID is the part between /d/ and /view  ->  1AbC...XyZ
# ------------------------------------------------------------------------------
LEGACY_DRIVE_ID = "PASTE_THE_GOOGLE_DRIVE_FILE_ID_HERE"

# Evaluate under both test-time settings so the comparison is unambiguous.
LEGACY_TTA = ["none", "hflip"]

from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
HF_REPO_ID = "Avi2006/spatial-moe-results"
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_REPO_ID"] = HF_REPO_ID

PROJECT_ROOT = "/kaggle/working/spatial_moe_sod"
CHECKPOINT_ROOT = "/kaggle/working/WXSOD_Checkpoints"
LEGACY_PATH = os.path.join(CHECKPOINT_ROOT, "legacy_best.pth")
LEGACY_OUT = "/kaggle/working/WXSOD_LegacyEval"
DATA_FILE_ID = "1SSELvRYI-cwd9mzA8dWLbv4o1IffjkoW"
EXTRACT_PATH = "/kaggle/working/WXSDO_data"

os.environ["CHECKPOINT_ROOT"] = CHECKPOINT_ROOT
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)

assert LEGACY_DRIVE_ID != "PASTE_THE_GOOGLE_DRIVE_FILE_ID_HERE", (
    "Paste the Google Drive file ID of the legacy checkpoint into LEGACY_DRIVE_ID."
)
print("Configuration")
print("  legacy checkpoint ->", LEGACY_PATH)
print("  evaluation output ->", LEGACY_OUT)
print("  TTA settings      ->", LEGACY_TTA)


## 02_dataset_acquire

Same acquisition path as the run notebooks: a Google Drive archive extracted locally, then the
real WXSOD root resolved by structure rather than by a hard-coded path.


In [ ]:
def is_dataset_valid(path):
    def check_root(r):
        reqs = ["train_sys/input", "train_sys/gt", "test_sys/input", "test_sys/gt",
                "test_real/input", "test_real/gt"]
        return all(os.path.isdir(os.path.join(r, req)) for req in reqs)

    if check_root(path):
        return path
    for root, dirs, _ in os.walk(path):
        if check_root(root):
            return root
    return None


valid_root = is_dataset_valid(EXTRACT_PATH)
if valid_root is None:
    print("Dataset not found locally, downloading from Google Drive...")
    ZIP_PATH = "/kaggle/working/dataset.zip"
    if not os.path.exists(ZIP_PATH):
        subprocess.run(["pip", "install", "-q", "gdown"], check=True)
        subprocess.run(["gdown", DATA_FILE_ID, "-O", ZIP_PATH], check=True)
    os.makedirs(EXTRACT_PATH, exist_ok=True)
    subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", EXTRACT_PATH], check=True)
    valid_root = is_dataset_valid(EXTRACT_PATH)

assert valid_root is not None, "Dataset extraction completed but no valid WXSOD root was found."
print("Dataset root:", valid_root)


## 03_project_deploy

The code comes from the Hub as a verified archive — the same route the run notebooks use, so
this evaluation runs the current implementation rather than whatever is already on the box.


In [ ]:
from huggingface_hub import hf_hub_download

os.makedirs(PROJECT_ROOT, exist_ok=True)
print(f"Downloading code from {os.environ['HF_REPO_ID']} ...")

manifest_path = hf_hub_download(
    repo_id=os.environ["HF_REPO_ID"], repo_type="dataset",
    filename="code/project_manifest.json", token=os.environ["HF_TOKEN"],
)
zip_path = hf_hub_download(
    repo_id=os.environ["HF_REPO_ID"], repo_type="dataset",
    filename="code/spatial_moe_sod_code.zip", token=os.environ["HF_TOKEN"],
)

with open(manifest_path, "r") as f:
    manifest = json.load(f)

sha256 = hashlib.sha256()
with open(zip_path, "rb") as f:
    for chunk in iter(lambda: f.read(4096), b""):
        sha256.update(chunk)

assert sha256.hexdigest() == manifest["archive_sha256"], (
    "Project source archive corrupted or stale - repackage and re-upload."
)

subprocess.run(["unzip", "-q", "-o", zip_path, "-d", PROJECT_ROOT], check=True)

required = ["src/train_ddp.py", "src/evaluate.py", "src/model.py", "src/experiment.py"]
for f in required:
    assert os.path.exists(os.path.join(PROJECT_ROOT, f)), f"Payload corrupted: missing {f}"

sys.path.insert(0, PROJECT_ROOT)
print("Project source deployed to:", PROJECT_ROOT)


## 04_dependencies

`src/evaluate.py` needs torch, torchvision, timm, albumentations, OpenCV, numpy and
`py_sod_metrics`. Missing ones are installed rather than assumed.


In [ ]:
import importlib

deps = {
    "torch": "torch",
    "torchvision": "torchvision",
    "timm": "timm",
    "albumentations": "albumentations",
    "opencv-python-headless": "cv2",
    "numpy": "numpy",
    "pysodmetrics": "py_sod_metrics",
}
for pip_name, mod_name in deps.items():
    try:
        mod = importlib.import_module(mod_name)
        print(f"  {pip_name:28s} {getattr(mod, '__version__', 'unknown')}")
    except ImportError:
        print(f"  {pip_name:28s} installing ...")
        subprocess.run(["pip", "install", "-q", pip_name], check=True)
print("Dependencies resolved.")


## 05_legacy_checkpoint

The legacy checkpoint is not tied to an experiment ID, so the run notebooks cannot resolve it.
It comes from Drive and is checked before use.


In [ ]:
EXPECTED_SHA256 = "8b7cac641afb1ff409c6e3db7caf2b826455bd1077158fd227b822e50d934ce0"

if not os.path.exists(LEGACY_PATH):
    print("Downloading the legacy checkpoint from Google Drive ...")
    subprocess.run(["pip", "install", "-q", "gdown"], check=True)
    subprocess.run(["gdown", LEGACY_DRIVE_ID, "-O", LEGACY_PATH], check=True)

assert os.path.exists(LEGACY_PATH), f"Legacy checkpoint missing: {LEGACY_PATH}"

h = hashlib.sha256()
with open(LEGACY_PATH, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
actual = h.hexdigest()
size_mb = os.path.getsize(LEGACY_PATH) / 1024 ** 2
print(f"  {LEGACY_PATH}  ({size_mb:.0f} MB)")
print(f"  sha256 {actual}")
if actual != EXPECTED_SHA256:
    print("  WARNING: sha256 does not match the value recorded on the Hub.")
    print("           Expected 8b7cac641afb1ff409c6e3db7caf2b826455bd1077158fd227b822e50d934ce0")
    print("           Continuing, but record which file was actually scored.")
else:
    print("  sha256 matches the value recorded on the Hub.")


## 06_identity

Prints the parameter-name fingerprint of the architecture this checkpoint creates. The 0.0168
evaluation recorded `2cd252ad3a3581e1c65961b828857d70`, so a different value means the file is
not provably that save — worth knowing when reading the metrics below.


In [ ]:
import torch
from src.evaluate import verify_parameter_consistency
from src.model import SpatialMoESODNet

ckpt = torch.load(LEGACY_PATH, map_location="cpu", weights_only=False)
print(f"  epoch={ckpt.get('epoch')}  global_step={ckpt.get('global_step')}  "
      f"best_metric={ckpt.get('best_metric')}")

mc = (ckpt.get("config") or {}).get("model") or {}
print(f"  config: experts={mc.get('num_experts')} k={mc.get('top_k')} "
      f"gate={mc.get('gate_mode', 'renormalized')} moe_type={mc.get('moe_type', 'sparse')} "
      f"deep_supervision={mc.get('deep_supervision')}")

probe = SpatialMoESODNet(
    dim=mc.get("working_dim", 256),
    num_experts=mc.get("num_experts", 8),
    k=mc.get("top_k", 2),
    gate_mode=mc.get("gate_mode", "renormalized"),
    window_size=mc.get("window_size", 8),
    moe_16_mode=mc.get("moe_16_mode", "sparse"),
    moe_type=mc.get("moe_type", "sparse"),
    use_deep_supervision=mc.get("deep_supervision", False),
    pretrained_backbone=False,
)
fingerprint = verify_parameter_consistency(probe)
print(f"  parameter-name fingerprint: {fingerprint}")
print(f"  recorded for the 0.0168 model: 2cd252ad3a3581e1c65961b828857d70")
print(f"  identical: {fingerprint == '2cd252ad3a3581e1c65961b828857d70'}")
del probe


## 07_evaluation

Both test-time settings, scored in original image coordinates. `none` is what the legacy
numbers were produced with; `hflip` is what the new runs use, so this run gives both sides of
the comparison.


In [ ]:
for tta in LEGACY_TTA:
    out_dir = os.path.join(LEGACY_OUT, tta)
    print("\n" + "=" * 70)
    print(f"Evaluating the legacy checkpoint with --tta {tta}")
    print("=" * 70)
    subprocess.run([
        "python", "-u", "-m", "src.evaluate",
        "--checkpoint", LEGACY_PATH,
        "--dataset", "both",
        "--data_dir", valid_root,
        "--out_dir", out_dir,
        "--tta", tta,
    ], cwd=PROJECT_ROOT, check=True)


## 08_results

Prints every metric produced, and restates the reference numbers so the diagnostic is readable
from the output alone.


In [ ]:
import glob

rows = []
for path in sorted(glob.glob(os.path.join(LEGACY_OUT, "**", "metrics_*.json"), recursive=True)):
    with open(path) as f:
        d = json.load(f)
    g = d.get("global", d)
    split = "test_real" if "test_real" in path else "test_sys"
    tta = "hflip" if "/hflip/" in path else "none"
    rows.append((tta, split, g.get("MAE"), g.get("S_measure"), g.get("E_adaptive"),
                 g.get("F_adaptive"), g.get("sample_count")))

print(f"  {'tta':6s} {'split':10s} {'MAE':>8s} {'S':>8s} {'E':>8s} {'F':>8s} {'n':>6s}")
for tta, split, mae, s, e, f, n in sorted(rows):
    print(f"  {tta:6s} {split:10s} {mae:8.4f} {s:8.4f} {e:8.4f} {f:8.4f} {n:6d}")

print("\nReference values (plain TTA, legacy evaluation):")
print("  test_real  MAE 0.0168  S 0.9151   (554 images)")
print("  test_sys   MAE 0.0192  S 0.9139   (1500 images)")
print("\nComparison values (hflip TTA, new runs, test_real):")
print("  REPRO 0.0195 | REPRO_SEED43 0.0199 | REPRO_SEED44 0.0191 | REPRODENSE 0.0199")
print("\nIf the plain-TTA row lands near 0.0168 the pipeline reproduces the legacy number;")
print("if it lands near 0.019 the evaluation protocol has changed since that measurement.")
